In [2]:
import tldextract

def extract_etld1(url):
    ext = tldextract.extract(url)
    return f"{ext.domain}.{ext.suffix}" if ext.suffix else ext.domain

In [6]:
def rbo(row,col1,col2,p=0.9):
    s, t = row[col1], row[col2]
    if not s or not t:
        return 0.0

    k = max(len(s), len(t))
    s_set, t_set = set(), set()
    rbo_sum = 0.0

    for d in range(1, k + 1):
        if d <= len(s):
            s_set.add(s[d - 1])
        if d <= len(t):
            t_set.add(t[d - 1])

        overlap = len(s_set & t_set)
        A_d = overlap / d
        rbo_sum += (p ** (d - 1)) * A_d

    A_k = len(s_set & t_set) / k
    return (1 - p) * rbo_sum + (p ** k) * A_k

In [7]:
import math

def relevance_from_reference(ref_list):
    return {
        item: 1 / math.log2(rank + 1) #len(ref_list)+1-rank
        for rank, item in enumerate(ref_list, start=1)
    }
def dcg(ranked_list, rel, k=None):
    if k is None:
        k = len(ranked_list)
    score = 0.0
    for i, item in enumerate(ranked_list[:k], start=1):
        score += rel.get(item, 0) / math.log2(i + 1)
    return score
def idcg(ref_list, rel, k=None):
    if k is None:
        k = len(ref_list)
    return dcg(ref_list, rel, k)
def ndcg_from_lists(row, col_comp, k=None):
    ref_list=row['links_serp']
    pred_list=row[col_comp]
    rel = relevance_from_reference(ref_list)
    if k is None:
        k=min(len(ref_list),len(pred_list))
    denom = idcg(ref_list, rel, k)
    if denom == 0:
        return 0.0
    return dcg(pred_list, rel, k) / denom


In [9]:
import pandas as pd
full_df=pd.read_csv('benchmark_dataset_links.csv')

import ast
import pandas as pd

def fix_list_column(x):
    # already a list → keep
    if isinstance(x, list):
        return x
    
    # missing / empty → empty list
    if pd.isna(x) or x == '' or x == 'nan':
        return []
    
    # string that looks like a list → parse
    if isinstance(x, str) and x.strip().startswith('['):
        try:
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return []
    
    # anything else → empty list
    return []

full_df["domains"] = full_df["domains"].apply(fix_list_column)
full_df["links"] = full_df["links"].apply(fix_list_column)
full_df["domains_serp"] = full_df["domains"].apply(fix_list_column)
full_df["links_serp"] = full_df["links"].apply(fix_list_column)
full_df["domains_aio"] = full_df["domains"].apply(fix_list_column)
full_df["links_aio"] = full_df["links"].apply(fix_list_column)
full_df

,Unnamed: 0_x,ID,query,dataset,label,AIO_12_7_8,has_keywords,length,length_2,length_z,length_z_bin,length_z_clipped,Unnamed: 0_y,links,domains,links_serp,domains_serp,links_aio,domains_aio
0,0,1,holidaytraditions,Amazon Retail,NaN,True,False,1,1,-1.849845,"(-3.4019999999999997, -1.384]",-1.849845,0,[https://forsythfamilymagazine.com/discovering...,"[forsythfamilymagazine.com, rutgers.edu, world...",[https://www.reddit.com/r/WitchesVsPatriarchy/...,"[reddit.com, buzzfeed.com, cozi.com, siriusxm....",[https://www.cozi.com/blog/28-holiday-traditio...,"[cozi.com, psychcentral.com, theeverymom.com, ..."
1,1,2,youtube camera for vlogging,Amazon Retail,NaN,True,False,4,4,0.199466,"(-0.0731, 0.367]",0.199466,1,"[https://store.insta360.com/vlogging, https://...","[insta360.com, provlogging.com, videomaker.com...",[https://www.bestbuy.com/site/digital-cameras/...,"[bestbuy.com, amazon.com, reddit.com, sony.com...",[https://www.youtube.com/watch?v=Ui9OXh-8QEI&t...,"[youtube.com, youtube.com, amateurphotographer..."
2,2,3,picasso tiles,Amazon Retail,NaN,True,False,2,2,-1.166741,"(-1.384, -0.953]",-1.166741,2,[https://littlecanadian.ca/blogs/baby101/magna...,"[littlecanadian.ca, celebratingwithkids.com, t...",[https://www.amazon.com/PicassoTiles-Construct...,"[amazon.com, picassotiles.com, picassotiles.co...",[https://www.picassotiles.com/collections/magn...,"[picassotiles.com, amazon.com, theeverymom.com..."
3,3,4,electronic bug detector,Amazon Retail,NaN,True,False,3,3,-0.483637,"(-0.517, -0.27]",-0.483637,3,[https://www.auscovertinvestigations.com.au/bu...,"[auscovertinvestigations.com.au, brickhousesec...",[https://www.spyguy.com/collections/bug-detect...,"[spyguy.com, brickhousesecurity.com, amazon.co...","[https://www.youtube.com/watch?v=oszyPhXo6Tg, ...","[youtube.com, amazon.com, brickhousesecurity.c..."
4,4,5,boxers for men,Amazon Retail,NaN,False,False,3,3,-0.483637,"(-0.517, -0.27]",-0.483637,4,[https://leonisa.uk/pages/diferentes-tipos-de-...,"[leonisa.uk, trendhim.co.uk, pantsandsocks.com...",[https://www.macys.com/shop/mens/clothing/unde...,"[macys.com, amazon.com, ethika.com, psd.com, h...",[],[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11495,11495,11496,twitch /me,ORCAS,Abstain,False,False,2,2,-0.764976,"(-0.953, -0.765]",-0.764976,11495,"[https://musicellatv.com/twitch-commands/, htt...","[musicellatv.com, dualshockers.com, youtube.co...",[https://www.reddit.com/r/Twitch/comments/ia83...,"[reddit.com, twitch.tv, youtube.com, twitch.tv...",[],[]
11496,11496,11497,notre dame map,ORCAS,Abstain,False,False,3,3,-0.146364,"(-0.176, -0.146]",-0.146364,11496,[https://www.michianaeye.com/storage/app/media...,"[michianaeye.com, forwardlook.net, simpleviewi...","[https://map.nd.edu/, https://assets.simplevie...","[nd.edu, simpleviewinc.com, ndm.edu, ndnj.org]",[],[]
11497,11497,11498,birthday word,ORCAS,Abstain,True,False,2,2,-0.764976,"(-0.953, -0.765]",-0.764976,11497,[https://www.paragraphai.com/post/happy-birthd...,"[paragraphai.com, wikihow.com, shutterfly.com,...",[https://www.shutterfly.com/ideas/happy-birthd...,"[shutterfly.com, grammarly.com, oed.com, cloud...",[https://www.southernliving.com/holidays-occas...,"[southernliving.com, paragraphai.com, parade.c..."
11498,11498,11499,aon brokers,ORCAS,Abstain,False,False,2,2,-0.764976,"(-0.953, -0.765]",-0.764976,11498,"[https://en.wikipedia.org/wiki/Aon_(company), ...","[wikipedia.org, aon.com, aon.com, aon.com, aon...",[https://www.aon.com/switzerland/test-chjk/pen...,"[aon.com, aon.com, aon.com, aonprograms.com, a...",[],[]


In [13]:
def remove_text_fragment(url):
    if isinstance(url, str):
        return url.split('#:~:text=')[0]
    return url

full_df["links_serp"] = full_df["links_serp"].apply(
    lambda lst: [remove_text_fragment(u) for u in lst] if isinstance(lst, list) else lst
)

full_df["links_aio"] = full_df["links_aio"].apply(
    lambda lst: [remove_text_fragment(u) for u in lst] if isinstance(lst, list) else lst
)

full_df["links"] = full_df["links"].apply(
    lambda lst: [remove_text_fragment(u) for u in lst] if isinstance(lst, list) else lst
)


In [22]:
full_df["rbo_aio_serp"] = full_df.apply(rbo,axis=1,col1='links_serp',col2='links_aio',p=0.9)
full_df["rbo_aio_gemini"] = full_df.apply(rbo,axis=1,col1='links',col2='links_aio',p=0.9)
full_df["rbo_gemini_serp"] = full_df.apply(rbo,axis=1,col1='links_serp',col2='links',p=0.9)

In [23]:
full_df["SERP_empty"] = full_df["domains_serp"].apply(lambda x: x == [])
full_df["AIO_empty"] = full_df["domains_aio"].apply(lambda x: x == [])
full_df["Gemini_empty"] = full_df["domains"].apply(lambda x: x == [])

In [44]:
len(full_df[(full_df.SERP_empty==False)&(full_df.AIO_empty==False)&(full_df.Gemini_empty==False)])

7439

In [24]:
#full_df["rbo_aio_serp"] = full_df.apply(rbo,axis=1,col1='domains_serp',col2='domains_aio',p=0.9)
#full_df["rbo_aio_gemini"] = full_df.apply(rbo,axis=1,col1='domains',col2='domains_aio',p=0.9)
#full_df["rbo_gemini_serp"] = full_df.apply(rbo,axis=1,col1='domains_serp',col2='domains',p=0.9)

In [25]:
full_df[(full_df.SERP_empty==False)&(full_df.AIO_empty==False)&(full_df.Gemini_empty==False)][['rbo_aio_serp','rbo_aio_gemini','rbo_gemini_serp']].mean()

rbo_aio_serp       0.227711
rbo_aio_gemini     0.149259
rbo_gemini_serp    0.212908
dtype: float64

In [26]:
print(full_df[(full_df.SERP_empty==False)&(full_df.AIO_empty==False)][['rbo_aio_serp']].mean())
print(full_df[(full_df.SERP_empty==False)&(full_df.Gemini_empty==False)][['rbo_gemini_serp']].mean())
print(full_df[(full_df.AIO_empty==False)&(full_df.Gemini_empty==False)][['rbo_aio_gemini']].mean())

rbo_aio_serp    0.227788
dtype: float64
rbo_gemini_serp    0.206903
dtype: float64
rbo_aio_gemini    0.149239
dtype: float64


In [27]:
full_df["ndcg_aio"] = full_df.apply(ndcg_from_lists,axis=1,col_comp='links_aio')
full_df["ndcg_gemini"] = full_df.apply(ndcg_from_lists,axis=1,col_comp='links')
full_df[(full_df.SERP_empty==False)&(full_df.AIO_empty==False)&(full_df.Gemini_empty==False)][['ndcg_gemini','ndcg_aio']].mean()

ndcg_gemini    0.309718
ndcg_aio       0.325975
dtype: float64

In [34]:
def jaccard_similarity(a, b):
    a_set = set(a)
    b_set = set(b)
    intersection = len(a_set & b_set)
    union = len(a_set | b_set)
    return intersection / union if union != 0 else 0

# Apply row-wise
full_df['jaccard_serp_aio'] = full_df.apply(lambda row: jaccard_similarity(row['links_aio'], row['links_serp']), axis=1)
full_df['jaccard_serp_gemini'] = full_df.apply(lambda row: jaccard_similarity(row['links'], row['links_serp']), axis=1)
full_df['jaccard_gemini_aio'] = full_df.apply(lambda row: jaccard_similarity(row['links_aio'], row['links']), axis=1)
full_df[(full_df.SERP_empty==False)&(full_df.AIO_empty==False)&(full_df.Gemini_empty==False)][['jaccard_gemini_aio','jaccard_serp_gemini','jaccard_serp_aio']].mean()

jaccard_gemini_aio     0.110783
jaccard_serp_gemini    0.163824
jaccard_serp_aio       0.177570
dtype: float64

In [32]:
full_df.jaccard_gemini_aio[0]

0.045454545454545456

In [176]:
print(full_df[(full_df.SERP_empty==False)&(full_df.Gemini_empty==False)][['ndcg_gemini']].mean())
print(full_df[(full_df.SERP_empty==False)&(full_df.AIO_empty==False)][['ndcg_aio']].mean())

ndcg_gemini    0.301276
dtype: float64
ndcg_aio    0.326204
dtype: float64


In [153]:
meta_max=0
full_df['ndcgs_gemini']=None
full_df['ndcgs_aio']=None
for index,row in full_df.iterrows():
    max_k_aio=min(len(row['links_aio']),len(row['links_serp']))
    max_k_gemini=min(len(row['links_serp']),len(row['links']))
    meta_max=max(max_k_aio,max_k_gemini,meta_max)
    if max_k_aio!=0:
        ndcgs_aio=[ndcg_from_lists(row,'links_aio',k) for k in range(1,max_k_aio+1)]
    else: 
        ndcgs_aio=None
    full_df.at[index,'ndcgs_aio']=ndcgs_aio
    if max_k_gemini!=0:
        ndcgs_gemini=[ndcg_from_lists(row,'links',k) for k in range(1,max_k_gemini+1)]
    else: 
        ndcgs_gemini=None
    full_df.at[index,'ndcgs_gemini']=ndcgs_gemini
print(meta_max)    
full_df

11


,Unnamed: 0_x,ID,query,dataset,label,AIO_12_7_8,has_keywords,length,length_2,length_z,...,rbo_aio_serp,rbo_aio_gemini,rbo_gemini_serp,SERP_empty,AIO_empty,Gemini_empty,ndcg_aio,ndcg_gemini,ndcgs_gemini,ndcgs_aio
0,0,1,holidaytraditions,Amazon Retail,NaN,True,False,1,1,-1.849845,...,0.129589,0.048342,0.074455,False,False,False,0.225103,0.075034,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0750343666551...","[0.5, 0.3576352816006229, 0.3033847384213055, ..."
1,1,2,youtube camera for vlogging,Amazon Retail,NaN,True,False,4,4,0.199466,...,0.044374,0.076361,0.095505,False,False,False,0.071817,0.109707,"[0.0, 0.0, 0.0, 0.07409832437086353, 0.0685067...","[0.0, 0.0, 0.10112824614043515, 0.090898117773..."
2,2,3,picasso tiles,Amazon Retail,NaN,True,False,2,2,-1.166741,...,0.110269,0.025299,0.071305,False,False,False,0.145207,0.071792,"[0.0, 0.0, 0.0, 0.0, 0.08400948806854203, 0.07...","[0.0, 0.0, 0.11736523773039913, 0.183787953095..."
3,3,4,electronic bug detector,Amazon Retail,NaN,True,False,3,3,-0.483637,...,0.000000,0.154127,0.061390,False,False,False,0.000000,0.054674,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.06013172961256673,...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
4,4,5,boxers for men,Amazon Retail,NaN,False,False,3,3,-0.483637,...,0.000000,0.000000,0.000000,False,True,False,0.000000,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]",None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11495,11495,11496,twitch /me,ORCAS,Abstain,False,False,2,2,-0.764976,...,0.000000,0.000000,0.223568,False,True,False,0.000000,0.322581,"[0.0, 0.0, 0.15169236921065274, 0.371233307648...",None
11496,11496,11497,notre dame map,ORCAS,Abstain,False,False,3,3,-0.146364,...,0.000000,0.000000,0.168355,False,True,False,0.000000,0.148197,"[0.0, 0.0, 0.0, 0.14819664874172706]",None
11497,11497,11498,birthday word,ORCAS,Abstain,True,False,2,2,-0.764976,...,0.000000,0.167114,0.303131,False,False,False,0.000000,0.425330,"[0.0, 0.0, 0.3033847384213055, 0.3563624813304...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
11498,11498,11499,aon brokers,ORCAS,Abstain,False,False,2,2,-0.764976,...,0.000000,0.000000,0.226848,False,True,False,0.000000,0.395742,"[0.3562071871080222, 0.2547845153390866, 0.216...",None


In [162]:
ndcg_gemini=[]
ndcg_aio=[]
for kk in range(meta_max):
    k=kk+1
    counter=0
    mysum=0
    for j in full_df['ndcgs_gemini']:
        if j is not None:
            try:
                mysum+=j[kk]
                counter+=1
            except IndexError:
                continue
    ndcg_gemini.append((mysum/counter,counter))
    
    counter2=0
    mysum2=0
    for j in full_df['ndcgs_aio']:
        if j is not None:
            try:
                mysum2+=j[kk]
                counter2+=1
            except IndexError:
                continue
    if counter2!=0:
        ndcg_aio.append((mysum2/counter2,counter2))
ndcg_gemini,ndcg_aio

([(0.2780371278314812, 11373),
  (0.2829220963280335, 11313),
  (0.2857775725924012, 11150),
  (0.2887549406610868, 10704),
  (0.28553480328547154, 9863),
  (0.2723538187451995, 8393),
  (0.2598032663886856, 7135),
  (0.24695674664274725, 5562),
  (0.23395629608778715, 3716),
  (0.22477689428321795, 1797),
  (0.6629438278719625, 1)],
 [(0.29769276864372013, 7483),
  (0.31383279246939794, 7428),
  (0.32054228889862174, 7294),
  (0.3238784989497324, 7070),
  (0.33027704457830365, 6693),
  (0.3337356870907611, 6209),
  (0.3364459685074395, 5501),
  (0.3442527881097826, 4536),
  (0.35349108283768815, 3177),
  (0.364959019375473, 1506)])

In [165]:
mysum=0
mycount=0
for j in full_df['ndcgs_gemini']:
    if j is not None:
        mysum+=j[-1]
        mycount+=1
    
mysum/mycount

0.3012762338453586

In [172]:
full_df[(full_df.SERP_empty==False)&(full_df.Gemini_empty==False)].groupby('dataset').mean('rbo_gemini_serp')[['rbo_gemini_serp','ndcg_gemini']]

,rbo_gemini_serp,ndcg_gemini
dataset,,
Amazon Retail,0.074080,0.094310
Amazon Retail Comp,0.102202,0.137730
Amazon Retail Q,0.123975,0.166827
Debate,0.142880,0.201198
ELI5,0.180492,0.256058
Localized,0.177758,0.259212
NQ,0.253006,0.405895
NQ keywords,0.258122,0.389255
ORCAS,0.244050,0.351628


In [173]:
full_df[(full_df.SERP_empty==False)&(full_df.AIO_empty==False)].groupby('dataset').mean('rbo_aio_serp')[['rbo_aio_serp','ndcg_aio']]

,rbo_aio_serp,ndcg_aio
dataset,,
Amazon Retail,0.118767,0.165411
Amazon Retail Comp,0.147932,0.205118
Amazon Retail Q,0.156688,0.216229
Debate,0.272268,0.368632
ELI5,0.247175,0.341539
Localized,0.168000,0.255729
NQ,0.235154,0.365418
NQ keywords,0.235612,0.345633
ORCAS,0.241660,0.346118


In [174]:
full_df[(full_df.Gemini_empty==False)&(full_df.AIO_empty==False)].groupby('dataset').mean('rbo_aio_gemini')[['rbo_aio_gemini']]

,rbo_aio_gemini
dataset,
Amazon Retail,0.100321
Amazon Retail Comp,0.101203
Amazon Retail Q,0.140561
Debate,0.118163
ELI5,0.144917
Localized,0.098086
NQ,0.173368
NQ keywords,0.162769
ORCAS,0.171248


In [36]:
full_df[(full_df.SERP_empty==False)&(full_df.Gemini_empty==False)&(full_df.AIO_empty==False)].groupby('dataset').mean('rbo_aio_gemini')[['jaccard_serp_aio','jaccard_gemini_aio','jaccard_serp_gemini','rbo_aio_serp','rbo_aio_gemini','rbo_gemini_serp','ndcg_aio','ndcg_gemini']]

,jaccard_serp_aio,jaccard_gemini_aio,jaccard_serp_gemini,rbo_aio_serp,rbo_aio_gemini,rbo_gemini_serp,ndcg_aio,ndcg_gemini
dataset,,,,,,,,
Amazon Retail,0.080598,0.064160,0.078117,0.118767,0.100321,0.096187,0.165411,0.122915
Amazon Retail Comp,0.110911,0.077608,0.083206,0.147571,0.101203,0.101882,0.204744,0.137940
Amazon Retail Q,0.123273,0.097326,0.100506,0.156688,0.140561,0.125129,0.216229,0.167943
Debate,0.240544,0.093777,0.110584,0.272268,0.118163,0.140984,0.368632,0.199054
ELI5,0.205161,0.110622,0.135541,0.247623,0.145072,0.182495,0.342087,0.259188
Localized,0.137337,0.073066,0.161846,0.166413,0.098086,0.208013,0.253944,0.302549
NQ,0.189957,0.132322,0.192795,0.235185,0.173368,0.258843,0.365516,0.414410
NQ keywords,0.187199,0.125138,0.197058,0.235462,0.162769,0.260031,0.345252,0.388384
ORCAS,0.169866,0.122412,0.202808,0.241576,0.171248,0.260939,0.345666,0.373214


In [43]:
from scipy.stats import wilcoxon

stat, p = wilcoxon(full_df[(full_df.dataset=='Amazon Retail')&(full_df.SERP_empty==False)&(full_df.Gemini_empty==False)&(full_df.AIO_empty==False)]['jaccard_serp_gemini'], full_df[(full_df.dataset=='Amazon Retail')&(full_df.SERP_empty==False)&(full_df.Gemini_empty==False)&(full_df.AIO_empty==False)]['jaccard_gemini_aio'])
print(stat,p)

1077.5 0.1333936802475055


In [99]:
len(full_df)

11500